# Fermeture $w_{sec}\partial_z\bar u$ / bulk-plume (Romps 2014) — UCLA-CRM RCE `large300`

Version corrigée : dimensions extraites par nom (peu importe l'ordre physique t/z/x/y réel du fichier), et **toutes les lectures se font par bloc de temps** (jamais niveau par niveau) — c'est ce qui règle le facteur 10 de lenteur observé avec les boucles `for iz in range(n_z)` sur les fichiers UCLA-CRM. Chaque bloc charge tous les niveaux d'un coup ; le découpage par niveau se fait ensuite en mémoire avec numpy, pas sur disque.

**À vérifier toi-même à l'exécution** (pas d'accès aux fichiers ici) : le print de contrôle en §0 (plage d'altitude, doit être en mètres croissants ~0-30000) et celui du PRW reconstruit (doit ressembler à 20-60 kg/m² en tropical).

## §0. Setup commun

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import xarray as xr
import gc, os
from scipy.ndimage import uniform_filter1d

plt.rcParams.update({'figure.dpi':120,'font.size':11,'axes.grid':True,
                     'grid.alpha':0.25,'image.cmap':'RdBu_r'})
Rd,Rv=287.05,461.5; EPSILON=Rd/Rv; g=9.81; cp=1004.0; p_ref=1e5
Z_TOP_KM=15.0

DIR = os.path.join(os.path.expanduser('~'), 'Desktop', 'UCLA-CRM')
def path3d(v): return os.path.join(DIR, f'UCLA-CRM_RCE_large300_3D_{v}.nc')
BLOC = 10   # taille de bloc temporel — augmente si tu as beaucoup de RAM, baisse si erreur memoire

_ds=xr.open_dataset(path3d('ua')); _da=_ds['ua']
dim_t,dim_z,dim_y,dim_x=_da.dims[:4]
n_t,n_z=_da.sizes[dim_t],_da.sizes[dim_z]; n_y,n_x=_da.sizes[dim_y],_da.sizes[dim_x]
alt=_ds[dim_z].values.astype(float).copy()
_ds.close(); del _ds,_da; gc.collect()
zkm=alt/1000.0
mask_show=alt<=Z_TOP_KM*1000; m28=(alt>=2000)&(alt<=8000); m812=(alt>=8000)&(alt<=12000)
m015=(alt>=0)&(alt<=2000); m212=(alt>=2000)&(alt<=12000); m112=(alt>=1000)&(alt<=12000)

t_stat=0; idx_stat=slice(t_stat,None); n_stat=n_t-t_stat
dt_phys=6*3600.0   # A VERIFIER : pas de temps entre snapshots 3D UCLA-CRM
tt=np.arange(n_stat)

def smooth_t(fld, win=8):
    return uniform_filter1d(fld, size=win, axis=1, mode='nearest')

def load_ordered(da, *dims_voulues):
    """Lit un DataArray dans son ordre natif (rapide), reordonne EN MEMOIRE ensuite."""
    ordre_natif = da.dims
    raw = da.values
    perm = [ordre_natif.index(d) for d in dims_voulues]
    return np.transpose(raw, perm)

print(f'dims : t={dim_t}, z={dim_z}, y={dim_y}, x={dim_x}')
print(f'altitude : min={alt.min():.1f} max={alt.max():.1f} (doit etre en metres, croissant, ~0-30000)')
print(f'{n_t}t x {n_z}z x {n_y}y x {n_x}x')


## §0bis. PRW reconstruit (aucune variable native) et masques humide/sec

In [ ]:
# ============================================================
#  PRW = (1/g) * integrale(hus dp), par bloc de temps
# ============================================================
prw_sum = np.zeros((n_y, n_x))
ds_hus = xr.open_dataset(path3d('hus')); ds_pa = xr.open_dataset(path3d('pa'))
for t0 in range(t_stat, n_t, BLOC):
    t1 = min(t0+BLOC, n_t); sl = {dim_t: slice(t0,t1)}
    qv = load_ordered(ds_hus['hus'].isel(sl), dim_t, dim_z, dim_y, dim_x)
    p  = load_ordered(ds_pa['pa'].isel(sl),   dim_t, dim_z, dim_y, dim_x)
    pwv_block = np.trapz(qv, p, axis=1)
    prw_sum += -pwv_block.sum(axis=0)/g
    del qv, p, pwv_block; gc.collect()
ds_hus.close(); ds_pa.close(); gc.collect()
prw_mean_arr = prw_sum / n_stat

PRW_SEUIL = float(np.median(prw_mean_arr.ravel()))
mh = (prw_mean_arr > PRW_SEUIL); ms = ~mh; mh_flat = mh.ravel(); ms_flat = ms.ravel()
f_h = float(mh.mean()); f_s = float(ms.mean())

print(f'PRW reconstruit : moyenne={prw_mean_arr.mean():.2f} kg/m2 (attendu ~20-60 en tropical)')
print(f'humide={f_h:.0%} sec={f_s:.0%}')


## §1. Champs de base — rho0, theta, T, ubar, wbar, flux de Reynolds, decomposition humide/sec

In [ ]:
# ============================================================
#  rho0(z), theta(z,t), T(z,t) — par bloc de temps, tous niveaux d'un coup
# ============================================================
rho0_sum=np.zeros(n_z); n_rho=0
theta_zt=np.zeros((n_z,n_stat)); T_zt=np.zeros((n_z,n_stat))
ds_ta=xr.open_dataset(path3d('ta')); ds_pa=xr.open_dataset(path3d('pa')); ds_hus=xr.open_dataset(path3d('hus'))
for t0 in range(t_stat,n_t,BLOC):
    t1=min(t0+BLOC,n_t); sl={dim_t:slice(t0,t1)}
    T = load_ordered(ds_ta['ta'].isel(sl),  dim_t, dim_z, dim_y, dim_x)
    p = load_ordered(ds_pa['pa'].isel(sl),  dim_t, dim_z, dim_y, dim_x)
    qv = load_ordered(ds_hus['hus'].isel(sl), dim_t, dim_z, dim_y, dim_x)
    Tv=T*(1+qv/EPSILON)/(1+qv); rho=p/(Rd*Tv)
    rho0_sum+=rho.mean(axis=(0,2,3))*(t1-t0); n_rho+=(t1-t0)
    theta = T*(p_ref/p)**(Rd/cp)
    theta_zt[:, t0:t1] = theta.mean(axis=(2,3)).T
    T_zt[:, t0:t1]     = T.mean(axis=(2,3)).T
    del T,p,qv,Tv,rho,theta; gc.collect()
ds_ta.close(); ds_pa.close(); ds_hus.close()
rho0=rho0_sum/n_rho
dthetadz_zt=np.gradient(theta_zt, alt, axis=0)
print('rho0, theta_zt, T_zt, dthetadz_zt : ok')


In [ ]:
# ============================================================
#  ubar_zt, wbar_zt, flux_zt (u'w') + u_h/s_zt, w_h/s_zt (humide/sec) — par bloc de temps
# ============================================================
ubar_zt=np.zeros((n_z,n_stat)); wbar_zt=np.zeros((n_z,n_stat)); flux_zt=np.zeros((n_z,n_stat))
u_h_zt=np.zeros((n_z,n_stat)); u_s_zt=np.zeros((n_z,n_stat))
w_h_zt=np.zeros((n_z,n_stat)); w_s_zt=np.zeros((n_z,n_stat))
ds_u=xr.open_dataset(path3d('ua')); ds_w=xr.open_dataset(path3d('wa'))
for t0 in range(t_stat,n_t,BLOC):
    t1=min(t0+BLOC,n_t); sl={dim_t:slice(t0,t1)}
    u = load_ordered(ds_u['ua'].isel(sl), dim_t, dim_z, dim_y, dim_x).reshape(t1-t0, n_z, -1)
    w = load_ordered(ds_w['wa'].isel(sl), dim_t, dim_z, dim_y, dim_x).reshape(t1-t0, n_z, -1)
    ub = u.mean(axis=2); wb = w.mean(axis=2)
    up = u - ub[:,:,None]; wp = w - wb[:,:,None]
    ubar_zt[:,t0:t1] = ub.T; wbar_zt[:,t0:t1] = wb.T
    flux_zt[:,t0:t1] = (rho0[None,:]*(up*wp).mean(axis=2)).T
    u_h_zt[:,t0:t1] = u[:,:,mh_flat].mean(axis=2).T
    u_s_zt[:,t0:t1] = u[:,:,ms_flat].mean(axis=2).T
    w_h_zt[:,t0:t1] = w[:,:,mh_flat].mean(axis=2).T
    w_s_zt[:,t0:t1] = w[:,:,ms_flat].mean(axis=2).T
    del u,w,ub,wb,up,wp; gc.collect()
ds_u.close(); ds_w.close(); gc.collect()

dudz_tot=np.gradient(ubar_zt, alt, axis=0)
uw_zt=flux_zt/rho0[:,None]
terme_flux=smooth_t(-(1/rho0[:,None])*np.gradient(flux_zt, alt, axis=0))
dudz_tot_s=smooth_t(dudz_tot); w_s_obs_s=smooth_t(w_s_zt)
print('ubar_zt, wbar_zt, flux_zt, u_h/s_zt, w_h/s_zt, terme_flux : ok')


In [ ]:
# ============================================================
#  tntr(z,t) — par bloc de temps
# ============================================================
tntr_zt=np.zeros((n_z,n_stat))
ds_tntr=xr.open_dataset(path3d('tntr'))
for t0 in range(t_stat,n_t,BLOC):
    t1=min(t0+BLOC,n_t); sl={dim_t:slice(t0,t1)}
    tr = load_ordered(ds_tntr['tntr'].isel(sl), dim_t, dim_z, dim_y, dim_x)
    tntr_zt[:,t0:t1] = tr.mean(axis=(2,3)).T
    del tr; gc.collect()
ds_tntr.close(); gc.collect()

with np.errstate(divide='ignore', invalid='ignore'):
    w_sec_rce_brut     = -tntr_zt/dthetadz_zt
    w_sec_rce_corrige  = -(theta_zt/T_zt)*tntr_zt/dthetadz_zt
w_sec_rce_brut_s    = smooth_t(w_sec_rce_brut)
w_sec_rce_corrige_s = smooth_t(w_sec_rce_corrige)
w_sec_rce_x_shear    = smooth_t(w_sec_rce_corrige*dudz_tot)
print('tntr_zt, w_sec_rce, fermeture radiative : ok')


## §2. Affichage final — fermeture radiative-subsidente sur 2–8 km

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(11,5),sharey=True)
v = max(np.percentile(np.abs(terme_flux[m28]),98), np.percentile(np.abs(w_sec_rce_x_shear[m28]),98))*3600+1e-12
for ax,(fld,lab) in zip(axes,[(w_sec_rce_x_shear*3600, r"$-(\theta/T)\dot T_{rad}/\partial_z\bar\theta\ \partial_z\bar u$ (m/s/h)"),
                                (terme_flux*3600, r"$-\frac{1}{\rho_0}\partial_z(\rho_0\overline{u'w'})$ (m/s/h)")]):
    im=ax.pcolormesh(tt+t_stat, zkm[m28], fld[m28], cmap='RdBu_r',
                      norm=TwoSlopeNorm(0,-v,v), shading='auto')
    ax.set_xlabel('t'); ax.set_title(lab,fontsize=10)
    fig.colorbar(im,ax=ax,pad=.02)
axes[0].set_ylabel('z (km)')
plt.suptitle('Fermeture radiative-subsidente vs flux reel (2-8km)', fontweight='bold')
plt.tight_layout(); plt.show()

r_brut = np.full(n_z, np.nan)
for iz in range(n_z):
    x, y = w_sec_rce_x_shear[iz,:], terme_flux[iz,:]
    if np.nanstd(x) < 1e-15 or np.nanstd(y) < 1e-15: continue
    r_brut[iz] = np.corrcoef(x,y)[0,1]
fig, ax = plt.subplots(figsize=(5,6))
ax.plot(r_brut[mask_show], zkm[mask_show], 'o-', ms=3); ax.axvline(0,color='k',lw=.5)
ax.set_xlabel('correlation'); ax.set_ylabel('z (km)'); ax.set_ylim(0,Z_TOP_KM)
ax.set_title('Correlation brute, sans coefficient ajuste')
plt.tight_layout(); plt.show()


## §3. Bulk-plume a la Romps (2014) — M, u_c, epsilon, D, jusqu'a 12km

In [ ]:
# ============================================================
#  M(z,t), u_c(z,t) — masque nuage (w>1m/s, clw+ice>1e-5), par bloc de temps
# ============================================================
SEUIL_W = 1.0; SEUIL_COND = 1e-5
N_MIN_POINTS = 200

M_zt = np.full((n_z, n_stat), np.nan)
uc_zt = np.full((n_z, n_stat), np.nan)
f_cloud_zt = np.zeros((n_z, n_stat))

ds_u = xr.open_dataset(path3d('ua')); ds_w = xr.open_dataset(path3d('wa'))
ds_clw = xr.open_dataset(path3d('clw')); ds_ice = xr.open_dataset(path3d('ice'))
for t0 in range(t_stat,n_t,BLOC):
    t1=min(t0+BLOC,n_t); sl={dim_t:slice(t0,t1)}
    u = load_ordered(ds_u['ua'].isel(sl), dim_t, dim_z, dim_y, dim_x).reshape(t1-t0, n_z, -1)
    w = load_ordered(ds_w['wa'].isel(sl), dim_t, dim_z, dim_y, dim_x).reshape(t1-t0, n_z, -1)
    clw = load_ordered(ds_clw['clw'].isel(sl), dim_t, dim_z, dim_y, dim_x).reshape(t1-t0, n_z, -1)
    ice = load_ordered(ds_ice['ice'].isel(sl), dim_t, dim_z, dim_y, dim_x).reshape(t1-t0, n_z, -1)
    cloud = (w > SEUIL_W) & ((clw+ice) > SEUIL_COND)
    for dt in range(t1-t0):
        it = t0+dt
        for iz in range(n_z):
            m = cloud[dt,iz,:]; f_cloud_zt[iz,it] = m.mean()
            if m.sum() >= N_MIN_POINTS:
                w_m = w[dt,iz,m]; u_m = u[dt,iz,m]
                M_zt[iz,it] = rho0[iz]*f_cloud_zt[iz,it]*w_m.mean()
                uc_zt[iz,it] = np.sum(w_m*u_m)/np.sum(w_m)
    del u,w,clw,ice,cloud; gc.collect()
ds_u.close(); ds_w.close(); ds_clw.close(); ds_ice.close(); gc.collect()

frac_valide = np.mean(np.isfinite(uc_zt[m212]))
print(f'fraction de points (z,t) valides sur 2-12km : {frac_valide:.0%}')
print(f'fraction nuageuse moyenne (2-8km) : {np.nanmean(f_cloud_zt[m28])*100:.1f}%')


In [ ]:
# ============================================================
#  comblement + lissage vertical de M et u_c
# ============================================================
def combler_et_lisser(champ_zt, fallback_zt, win=5):
    out = champ_zt.copy()
    for it in range(n_stat):
        col = out[:,it]; valid = np.isfinite(col)
        if valid.sum() < 3:
            out[:,it] = fallback_zt[:,it]; continue
        out[:,it] = np.interp(alt, alt[valid], col[valid])
    return uniform_filter1d(out, size=win, axis=0, mode='nearest')

M_zt_lisse  = combler_et_lisser(M_zt,  np.zeros_like(M_zt))
uc_zt_lisse = combler_et_lisser(uc_zt, ubar_zt)

duc_dz_zt = np.gradient(uc_zt_lisse, alt, axis=0)
denom_zt = ubar_zt - uc_zt_lisse
echelle = np.nanpercentile(np.abs(denom_zt[m212]), 90)
with np.errstate(divide='ignore', invalid='ignore'):
    eps_zt = np.where(np.abs(denom_zt) > 0.3*echelle, duc_dz_zt/denom_zt, np.nan)

dMdz_zt = np.gradient(M_zt_lisse, alt, axis=0)
echelle_M = np.nanpercentile(np.abs(M_zt_lisse[m212]), 90)
with np.errstate(divide='ignore', invalid='ignore'):
    D_zt = np.where(np.abs(M_zt_lisse) > 0.3*echelle_M, eps_zt - dMdz_zt/M_zt_lisse, np.nan)

eps_z_composite = np.nanmedian(eps_zt, axis=1)
D_z_composite = np.nanmedian(D_zt, axis=1)

fig, ax = plt.subplots(figsize=(6,8))
ax.plot(eps_z_composite[mask_show]*1e3, zkm[mask_show], 'o-', ms=3, label='epsilon')
ax.plot(D_z_composite[mask_show]*1e3, zkm[mask_show], 'o-', ms=3, label='D')
ax.axhline(12, color='r', lw=1, ls='--'); ax.axvline(0,color='k',lw=.5); ax.legend(fontsize=9)
ax.set_xlabel(r'($10^{-3}$ m$^{-1}$)'); ax.set_ylabel('z (km)')
ax.set_title('Entrainement et detrainement, champs lisses')
plt.tight_layout(); plt.show()


## §4. Test direct de la prediction bulk-plume, jusqu'a 12km

In [ ]:
dudt_pred = (1/rho0[:,None]) * np.gradient(M_zt_lisse*(ubar_zt - uc_zt_lisse), alt, axis=0)
dudt_pred_s = smooth_t(dudt_pred)
dudt_reel = smooth_t(np.gradient(ubar_zt, dt_phys, axis=1))

fig, axes = plt.subplots(1,2,figsize=(11,5),sharey=True)
v = max(np.nanpercentile(np.abs(dudt_reel[m212]),95), np.nanpercentile(np.abs(dudt_pred_s[m212]),95))*3600+1e-12
for ax,(fld,lab) in zip(axes,[(dudt_reel*3600, r"$\partial_t\bar u$ reel (m/s/h)"),
                                (dudt_pred_s*3600, r"$\frac{1}{\rho_0}\partial_z[M(\bar u-u_c)]$ predit (m/s/h)")]):
    im=ax.pcolormesh(tt+t_stat, zkm[m212], fld[m212], cmap='RdBu_r',
                      norm=TwoSlopeNorm(0,-v,v), shading='auto')
    ax.set_xlabel('t'); ax.set_title(lab,fontsize=10)
    fig.colorbar(im,ax=ax,pad=.02)
axes[0].set_ylabel('z (km)')
plt.suptitle('Bulk-plume (Romps) — 2-12km', fontweight='bold')
plt.tight_layout(); plt.show()

R2_romps = np.full(n_z, np.nan)
for iz in range(n_z):
    if not m212[iz]: continue
    a,b = dudt_reel[iz,:], dudt_pred_s[iz,:]
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum()<20 or np.std(a[ok])<1e-15: continue
    ss_res=np.sum((a[ok]-b[ok])**2); ss_tot=np.sum((a[ok]-a[ok].mean())**2)
    R2_romps[iz]=1-ss_res/ss_tot if ss_tot>0 else np.nan

fig, ax = plt.subplots(figsize=(5,7))
ax.plot(R2_romps[m212], zkm[m212], 'o-', ms=3); ax.axvline(0,color='k',lw=.5)
ax.axhline(10, color='r', lw=1, ls='--', label='limite theorique Romps (overshoot)')
ax.set_xlabel('R2'); ax.set_ylabel('z (km)'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print(f'R2 median 2-8km  : {np.nanmedian(R2_romps[m28]):.2f}')
print(f'R2 median 8-12km : {np.nanmedian(R2_romps[m812]):.2f}')


## §5. Comparaison finale — bulk-plume vs fermeture radiative, correlation et R2 recalibre

In [ ]:
# ============================================================
#  Trois cartes cote a cote : dudt reel | bulk-plume | radiative
# ============================================================
w_sec_rad = (theta_zt/T_zt) * tntr_zt / dthetadz_zt
dudt_pred_rad = smooth_t(-w_sec_rad * dudz_tot)

fig, axes = plt.subplots(1,3,figsize=(16,5),sharey=True)
v = max(np.nanpercentile(np.abs(dudt_reel[m112]),95),
        np.nanpercentile(np.abs(dudt_pred_s[m112]),95),
        np.nanpercentile(np.abs(dudt_pred_rad[m112]),95))*3600+1e-12
for ax,(fld,lab) in zip(axes,[(dudt_reel*3600, r"$\partial_t\bar u$ reel (m/s/h)"),
                                (dudt_pred_s*3600, r"$\frac{1}{\rho_0}\partial_z[M(\bar u-u_c)]$ (m/s/h)"),
                                (dudt_pred_rad*3600, r"$-\frac{\dot Q_{rad}}{\partial_z\bar\theta}\partial_z\bar u$ (m/s/h)")]):
    im=ax.pcolormesh(tt+t_stat, zkm[m112], fld[m112], cmap='RdBu_r',
                      norm=TwoSlopeNorm(0,-v,v), shading='auto')
    ax.set_xlabel('t'); ax.set_title(lab,fontsize=10)
    fig.colorbar(im,ax=ax,pad=.02)
axes[0].set_ylabel('z (km)')
plt.suptitle('dudt reel vs deux predictions (1-12km)', fontweight='bold')
plt.tight_layout(); plt.show()

# ---- correlation ----
corr_bulk = np.full(n_z, np.nan); corr_rad = np.full(n_z, np.nan)
for iz in range(n_z):
    if not m112[iz]: continue
    a = dudt_reel[iz,:]
    for pred, store in [(dudt_pred_s[iz,:], 'bulk'), (dudt_pred_rad[iz,:], 'rad')]:
        ok = np.isfinite(a) & np.isfinite(pred)
        if ok.sum() < 20 or np.std(a[ok]) < 1e-15 or np.std(pred[ok]) < 1e-15: continue
        r = np.corrcoef(a[ok], pred[ok])[0,1]
        if store == 'bulk': corr_bulk[iz] = r
        else: corr_rad[iz] = r

fig, axes = plt.subplots(1,2,figsize=(10,7),sharey=True)
axes[0].plot(corr_bulk[m112], zkm[m112], 'o-', ms=3); axes[0].axvline(0,color='k',lw=.5)
axes[0].set_title('correlation bulk-plume'); axes[0].set_xlabel('r'); axes[0].set_ylabel('z (km)')
axes[1].plot(corr_rad[m112], zkm[m112], 'o-', ms=3); axes[1].axvline(0,color='k',lw=.5)
axes[1].set_title('correlation radiative'); axes[1].set_xlabel('r')
plt.suptitle('Correlation a dudt_reel', fontweight='bold')
plt.tight_layout(); plt.show()

# ---- R2 recalibre (pente+biais) ----
def r2_recalibre(pred):
    pente = np.full(n_z, np.nan); biais = np.full(n_z, np.nan)
    for iz in range(n_z):
        if not m112[iz]: continue
        a, b = dudt_reel[iz,:], pred[iz,:]
        ok = np.isfinite(a) & np.isfinite(b)
        if ok.sum() < 20 or np.std(b[ok]) < 1e-15: continue
        A = np.vstack([b[ok], np.ones(ok.sum())]).T
        coefs, *_ = np.linalg.lstsq(A, a[ok], rcond=None)
        pente[iz], biais[iz] = coefs
    pred_cal = pente[:,None]*pred + biais[:,None]
    r2 = np.full(n_z, np.nan)
    for iz in range(n_z):
        if not m112[iz]: continue
        a, b = dudt_reel[iz,:], pred_cal[iz,:]
        ok = np.isfinite(a) & np.isfinite(b)
        if ok.sum() < 20 or np.std(a[ok]) < 1e-15: continue
        ss_res = np.sum((a[ok]-b[ok])**2); ss_tot = np.sum((a[ok]-a[ok].mean())**2)
        r2[iz] = 1-ss_res/ss_tot if ss_tot>0 else np.nan
    return r2

R2_bulk_cal = r2_recalibre(dudt_pred_s)
R2_rad_cal = r2_recalibre(dudt_pred_rad)

fig, axes = plt.subplots(1,2,figsize=(10,7),sharey=True)
axes[0].plot(R2_bulk_cal[m112], zkm[m112], 'o-', ms=3); axes[0].axvline(0,color='k',lw=.5)
axes[0].set_title('R2 recalibre, bulk-plume'); axes[0].set_xlabel('R2'); axes[0].set_ylabel('z (km)')
axes[1].plot(R2_rad_cal[m112], zkm[m112], 'o-', ms=3); axes[1].axvline(0,color='k',lw=.5)
axes[1].set_title('R2 recalibre, radiatif'); axes[1].set_xlabel('R2')
plt.suptitle('R2 recalibres (pente+biais par niveau)', fontweight='bold')
plt.tight_layout(); plt.show()

print(f"R2 median bulk-plume (2-8km / 8-12km) : {np.nanmedian(R2_bulk_cal[m28]):.2f} / {np.nanmedian(R2_bulk_cal[m812]):.2f}")
print(f"R2 median radiatif   (2-8km / 8-12km) : {np.nanmedian(R2_rad_cal[m28]):.2f} / {np.nanmedian(R2_rad_cal[m812]):.2f}")
